# 3\.2\.2 Explore Pre/Post Mobility Structure Shifts

This notebook builds on the global PCA analysis performed in Notebook 3\.2\.1\. The previous notebook showed that pre\- and post\-congestion\-pricing observations largely overlap in shared PCA score space across all transportation systems, suggesting that congestion pricing does not create a simple global separation in the leading principal components\. Rather than revisiting PCA compressibility, score\-space organization, or component interpretation, this notebook focuses on a narrower question: whether the dominant mobility relationships captured by PCA remain stable before and after congestion pricing implementation\.

The primary goal is to determine whether congestion pricing altered the underlying geometry of the mobility\-state feature space\. To answer this question, we fit separate PCA models for pre\- and post\-congestion\-pricing observations and compare their variance structure, principal\-component loading patterns, and leading feature drivers\. If the resulting PCA structures are highly similar, this would suggest that the same latent mobility processes continue to organize the feature space across both policy regimes\. Conversely, substantial differences would indicate that congestion pricing may have altered the dominant relationships between mobility features even if observations continue to occupy similar regions of the overall feature space\.

In [1]:
# ---------------------------------------------------------------------
# Imports and notebook display settings
# ---------------------------------------------------------------------

from pathlib import Path
import pickle
import warnings

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.decomposition import PCA

from project_branding import (
    BRAND_COLORS,
    BRAND_COLOR_SEQUENCE,
    BRAND_DIVERGING_SEQUENCE,
)

warnings.filterwarnings("ignore")

px.defaults.color_discrete_sequence = BRAND_COLOR_SEQUENCE
px.defaults.template = "plotly_white"

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)

print("Notebook setup imports loaded.")

Notebook setup imports loaded.


In [2]:
# ---------------------------------------------------------------------
# Project paths and PCA stability settings
# ---------------------------------------------------------------------

PIPELINE_DATA_DIR = Path("pipeline_data")

INPUT_MATRIX_DIR = PIPELINE_DATA_DIR / "3.1.1.final_tables"
OUTPUT_DIR = PIPELINE_DATA_DIR / "3.2.2.final_tables"
INTERMEDIATE_OUTPUT_DIR = PIPELINE_DATA_DIR / "3.2.2.intermediate_tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_MODELING_ASSET_MANIFEST_PATH = INPUT_MATRIX_DIR / "final_modeling_asset_manifest.csv"

MODEL_FEATURE_SET_NAMES = [
    "bus",
    "subway",
    "taxi",
    "fhvhv",
    "multimodal",
]

PCA_REPRESENTATION_POLICY_BY_SET = {
    "bus": "full",
    "subway": "full",
    "taxi": "mobility_only",
    "fhvhv": "mobility_only",
    "multimodal": "mobility_only",
}

POLICY_PERIOD_VALUES = ["pre_cp", "post_cp"]
MISSING_INDICATOR_SUFFIX = "_missing_indicator"

PCA_VARIANCE_THRESHOLDS = [0.80, 0.90, 0.95]
PCA_COMPONENTS_TO_COMPARE = 3
PCA_TOP_FEATURE_COUNT = 10
PCA_RANDOM_STATE = 42

REBUILD_PERIOD_PCA_MODELS = False
REBUILD_TAXI_PERIOD_PCA_EXPORTS = False

PCA_PERIOD_MODEL_CACHE_DIR = INTERMEDIATE_OUTPUT_DIR / "pca_period_models"
PCA_PERIOD_MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input matrix directory: {INPUT_MATRIX_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Feature sets: {', '.join(MODEL_FEATURE_SET_NAMES)}")

Input matrix directory: pipeline_data/3.1.1.final_tables
Output directory: pipeline_data/3.2.2.final_tables
Feature sets: bus, subway, taxi, fhvhv, multimodal


In [3]:
# ---------------------------------------------------------------------
# Shared helpers
# ---------------------------------------------------------------------

def status_label(condition):
    return "pass" if condition else "review"


def sql_path(path):
    return str(path).replace("\\", "/").replace("'", "''")


def duckdb_identifier(column_name):
    escaped_name = str(column_name).replace('"', '""')
    return f'"{escaped_name}"'


def read_parquet_schema(parquet_path):
    parquet_path_sql = sql_path(parquet_path)
    with duckdb.connect() as con:
        return con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{parquet_path_sql}')"
        ).fetchdf()


def get_parquet_row_count(parquet_path):
    parquet_path_sql = sql_path(parquet_path)
    with duckdb.connect() as con:
        return con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{parquet_path_sql}')"
        ).fetchone()[0]


def identify_model_columns(parquet_path):
    schema_df = read_parquet_schema(parquet_path)
    return [
        column
        for column in schema_df["column_name"].tolist()
        if column != "modeling_row_id"
    ]


def is_missing_indicator_column(column_name):
    return str(column_name).endswith(MISSING_INDICATOR_SUFFIX)


def components_for_threshold(cumulative_variance_ratio, threshold):
    return int(np.argmax(cumulative_variance_ratio >= threshold) + 1)


def sign_aligned_similarity(vector_a, vector_b):
    cosine_similarity = np.dot(vector_a, vector_b) / (
        np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    )
    sign = 1 if cosine_similarity >= 0 else -1
    aligned_vector_b = vector_b * sign
    aligned_cosine = abs(cosine_similarity)
    aligned_correlation = abs(np.corrcoef(vector_a, vector_b)[0, 1])
    return aligned_cosine, aligned_correlation, sign, aligned_vector_b


print("Shared helpers ready.")

Shared helpers ready.


## 3\.2\.2\.1 Load Modeling Assets and Create Pre/Post Splits

In this section, we load the canonical scaled modeling matrices produced in Notebook 3\.1\.1 and apply the default PCA representation policy established in Notebook 3\.2\.1\. Bus and Subway use their full feature sets, while Taxi, FHVHV, and Multimodal use mobility\-only feature inputs with missingness indicators excluded\. Each modality is then partitioned into pre\-congestion\-pricing and post\-congestion\-pricing subsets using the canonical pre\_post\_cp field\. These subsets will serve as the inputs for all subsequent PCA stability analyses\.

Start by loading the 3\.1\.1 asset manifest and creating path lookups for the scaled matrices and row metadata tables\. These are the only source assets this notebook needs\.

In [4]:
# ---------------------------------------------------------------------
# Load 3.1.1 matrix asset paths
# ---------------------------------------------------------------------

final_modeling_asset_manifest_df = pd.read_csv(FINAL_MODELING_ASSET_MANIFEST_PATH)

matrix_asset_rows_df = final_modeling_asset_manifest_df.loc[
    final_modeling_asset_manifest_df["feature_set"].isin(MODEL_FEATURE_SET_NAMES)
    & final_modeling_asset_manifest_df["asset_group"].isin(["scaled_matrix", "row_metadata"])
].copy()

matrix_asset_path_df = (
    matrix_asset_rows_df
    .pivot_table(
        index="feature_set",
        columns="asset_group",
        values="path",
        aggfunc="first",
    )
    .reset_index()
    .rename(
        columns={
            "scaled_matrix": "scaled_matrix_path",
            "row_metadata": "row_metadata_path",
        }
    )
)

matrix_asset_path_df["scaled_exists"] = matrix_asset_path_df["scaled_matrix_path"].map(lambda path: Path(path).exists())
matrix_asset_path_df["metadata_exists"] = matrix_asset_path_df["row_metadata_path"].map(lambda path: Path(path).exists())
matrix_asset_path_df["status"] = np.where(
    matrix_asset_path_df["scaled_exists"] & matrix_asset_path_df["metadata_exists"],
    "pass",
    "review",
)

SCALED_MATRIX_PATHS_BY_SET = {
    row["feature_set"]: Path(row["scaled_matrix_path"])
    for _, row in matrix_asset_path_df.iterrows()
}

ROW_METADATA_PATHS_BY_SET = {
    row["feature_set"]: Path(row["row_metadata_path"])
    for _, row in matrix_asset_path_df.iterrows()
}

display(
    matrix_asset_path_df[
        ["feature_set", "scaled_exists", "metadata_exists", "status"]
    ].sort_values("feature_set")
)

assert matrix_asset_path_df["status"].eq("pass").all(), (
    "One or more 3.1.1 scaled matrix or metadata assets is missing."
)

print("3.1.1 matrix asset paths loaded.")

asset_group,feature_set,scaled_exists,metadata_exists,status
0,bus,True,True,pass
1,fhvhv,True,True,pass
2,multimodal,True,True,pass
3,subway,True,True,pass
4,taxi,True,True,pass


3.1.1 matrix asset paths loaded.


Findings\. The required scaled matrix and metadata assets are present for all five feature sets, so the notebook can use the finalized 3\.1\.1 handoff without rebuilding inputs\.

Apply the 3\.2\.1 default PCA representation policy\. Bus and Subway keep all scaled columns; Taxi, FHVHV, and Multimodal drop missingness indicators for PCA stability testing\.

In [5]:
# ---------------------------------------------------------------------
# Prepare PCA input columns by representation policy
# ---------------------------------------------------------------------

MODEL_COLUMNS_BY_SET = {}
PCA_INPUT_COLUMNS_BY_SET = {}
EXCLUDED_INDICATOR_COLUMNS_BY_SET = {}

pca_input_column_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    model_columns = identify_model_columns(SCALED_MATRIX_PATHS_BY_SET[feature_set_name])
    indicator_columns = [
        column for column in model_columns
        if is_missing_indicator_column(column)
    ]

    representation_policy = PCA_REPRESENTATION_POLICY_BY_SET[feature_set_name]

    if representation_policy == "mobility_only":
        pca_input_columns = [
            column for column in model_columns
            if column not in indicator_columns
        ]
        excluded_indicator_columns = indicator_columns
    else:
        pca_input_columns = model_columns
        excluded_indicator_columns = []

    MODEL_COLUMNS_BY_SET[feature_set_name] = model_columns
    PCA_INPUT_COLUMNS_BY_SET[feature_set_name] = pca_input_columns
    EXCLUDED_INDICATOR_COLUMNS_BY_SET[feature_set_name] = excluded_indicator_columns

    pca_input_column_records.append(
        {
            "feature_set": feature_set_name,
            "representation_policy": representation_policy,
            "scaled_model_columns": len(model_columns),
            "pca_input_columns": len(pca_input_columns),
            "excluded_indicator_columns": len(excluded_indicator_columns),
        }
    )

pca_input_column_summary_df = pd.DataFrame(pca_input_column_records)

display(pca_input_column_summary_df)

print("PCA input columns prepared.")

,feature_set,representation_policy,scaled_model_columns,pca_input_columns,excluded_indicator_columns
0,bus,full,40,40,0
1,subway,full,21,21,0
2,taxi,mobility_only,53,38,15
3,fhvhv,mobility_only,46,37,9
4,multimodal,mobility_only,233,142,91


PCA input columns prepared.


Findings\. The PCA representation policy matches the 3\.2\.1 decision: Bus and Subway use full feature sets, while Taxi, FHVHV, and Multimodal use mobility\-only inputs that exclude missingness indicators\.

Create the pre/post split summary from row metadata\. This does not write new split matrices; it confirms the available observation universe for each period and keeps later PCA fits pointed at the existing 3\.1\.1 assets\.

In [6]:
# ---------------------------------------------------------------------
# Summarize pre/post row counts by feature set
# ---------------------------------------------------------------------

period_split_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    matrix_path = SCALED_MATRIX_PATHS_BY_SET[feature_set_name]
    metadata_path = ROW_METADATA_PATHS_BY_SET[feature_set_name]

    matrix_path_sql = sql_path(matrix_path)
    metadata_path_sql = sql_path(metadata_path)

    with duckdb.connect() as con:
        period_counts_df = con.execute(
            f"""
            SELECT
                '{feature_set_name}' AS feature_set,
                m.pre_post_cp,
                COUNT(*) AS rows
            FROM read_parquet('{matrix_path_sql}') AS x
            INNER JOIN read_parquet('{metadata_path_sql}') AS m
                USING (modeling_row_id)
            GROUP BY 1, 2
            ORDER BY 1, 2
            """
        ).fetchdf()

    period_split_records.append(period_counts_df)

period_split_summary_df = pd.concat(period_split_records, ignore_index=True)

period_split_display_df = (
    period_split_summary_df
    .pivot_table(
        index="feature_set",
        columns="pre_post_cp",
        values="rows",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

for period_value in POLICY_PERIOD_VALUES:
    if period_value not in period_split_display_df.columns:
        period_split_display_df[period_value] = 0

period_split_display_df["total_rows"] = period_split_display_df[POLICY_PERIOD_VALUES].sum(axis=1)
period_split_display_df["post_share"] = (
    period_split_display_df["post_cp"]
    / period_split_display_df["total_rows"].replace(0, np.nan)
).round(3)
period_split_display_df["status"] = np.where(
    period_split_display_df[POLICY_PERIOD_VALUES].gt(0).all(axis=1),
    "pass",
    "review",
)

period_split_display_df = period_split_display_df[
    ["feature_set", "pre_cp", "post_cp", "total_rows", "post_share", "status"]
].sort_values("feature_set")

display(period_split_display_df)

assert period_split_display_df["status"].eq("pass").all(), (
    "One or more feature sets is missing a pre_cp or post_cp split."
)

print("Pre/post row-count split summarized.")

pre_post_cp,feature_set,pre_cp,post_cp,total_rows,post_share,status
0,bus,912228,560719,1472947,0.381,pass
1,fhvhv,955500,586300,1541800,0.380,pass
2,multimodal,955500,586300,1541800,0.380,pass
3,subway,569545,341910,911455,0.375,pass
4,taxi,955500,586300,1541800,0.380,pass


Pre/post row-count split summarized.


Findings\. All five feature sets contain both pre\_cp and post\_cp observations, with post\_cp representing roughly 38% of rows, so the period\-specific PCA comparisons have enough data on both sides\.

## 3\.2\.2\.2 Compare PCA Variance Structure Across Policy Periods

Notebook 3\.2\.1 showed that each mobility feature space has interpretable PCA structure: single\-modality matrices compress reasonably well, while the Multimodal space is broader but still organized around meaningful mobility relationships\. This section checks whether that broad variance structure is similar before and after congestion pricing\. If PC1, PC2, first\-10 cumulative variance, and common variance thresholds remain close across periods, then the feature space has similar overall complexity in both regimes\.

Fit separate PCA models for pre\_cp and post\_cp observations using the default PCA input columns from the previous section\. The output summarizes early\-component variance so we can compare broad feature\-space complexity by period\.

In [7]:
# ---------------------------------------------------------------------
# Fit or load pre/post PCA models by feature set
# ---------------------------------------------------------------------

PERIOD_PCA_MODELS_BY_SET = {}
PERIOD_PCA_EXPLAINED_VARIANCE_BY_SET = {}
PERIOD_PCA_CUMULATIVE_VARIANCE_BY_SET = {}

pca_period_fit_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    PERIOD_PCA_MODELS_BY_SET[feature_set_name] = {}
    PERIOD_PCA_EXPLAINED_VARIANCE_BY_SET[feature_set_name] = {}
    PERIOD_PCA_CUMULATIVE_VARIANCE_BY_SET[feature_set_name] = {}

    matrix_path = SCALED_MATRIX_PATHS_BY_SET[feature_set_name]
    metadata_path = ROW_METADATA_PATHS_BY_SET[feature_set_name]
    model_columns = PCA_INPUT_COLUMNS_BY_SET[feature_set_name]

    selected_columns_sql = ", ".join(
        duckdb_identifier(column) for column in model_columns
    )
    matrix_path_sql = sql_path(matrix_path)
    metadata_path_sql = sql_path(metadata_path)

    for period_value in POLICY_PERIOD_VALUES:
        model_cache_path = (
            PCA_PERIOD_MODEL_CACHE_DIR
            / f"{feature_set_name}_{period_value}_pca_model.pkl"
        )

        pca_model = None
        fit_status = "fit"

        if model_cache_path.exists() and not REBUILD_PERIOD_PCA_MODELS:
            with open(model_cache_path, "rb") as model_file:
                cached_model = pickle.load(model_file)

            cached_feature_names = list(getattr(cached_model, "feature_names_in_", []))

            if cached_feature_names == model_columns:
                pca_model = cached_model
                fit_status = "loaded_from_cache"

        if pca_model is None:
            with duckdb.connect() as con:
                X = con.execute(
                    f"""
                    SELECT {selected_columns_sql}
                    FROM read_parquet('{matrix_path_sql}') AS x
                    INNER JOIN read_parquet('{metadata_path_sql}') AS m
                        USING (modeling_row_id)
                    WHERE m.pre_post_cp = '{period_value}'
                    """
                ).fetchdf()

            X = X.astype("float32", copy=False)

            pca_model = PCA(
                n_components=len(model_columns),
                svd_solver="auto",
                random_state=PCA_RANDOM_STATE,
            )
            pca_model.fit(X)

            with open(model_cache_path, "wb") as model_file:
                pickle.dump(pca_model, model_file)

            row_count = len(X)
            del X
        else:
            row_count = int(
                period_split_summary_df.loc[
                    period_split_summary_df["feature_set"].eq(feature_set_name)
                    & period_split_summary_df["pre_post_cp"].eq(period_value),
                    "rows",
                ].iloc[0]
            )

        explained_variance_ratio = pca_model.explained_variance_ratio_
        cumulative_variance_ratio = np.cumsum(explained_variance_ratio)

        PERIOD_PCA_MODELS_BY_SET[feature_set_name][period_value] = pca_model
        PERIOD_PCA_EXPLAINED_VARIANCE_BY_SET[feature_set_name][period_value] = explained_variance_ratio
        PERIOD_PCA_CUMULATIVE_VARIANCE_BY_SET[feature_set_name][period_value] = cumulative_variance_ratio

        pca_period_fit_records.append(
            {
                "feature_set": feature_set_name,
                "policy_period": period_value,
                "rows": row_count,
                "features": len(model_columns),
                "pc1_variance_pct": explained_variance_ratio[0] * 100,
                "pc2_variance_pct": explained_variance_ratio[1] * 100,
                "pc10_cumulative_variance_pct": cumulative_variance_ratio[min(9, len(model_columns) - 1)] * 100,
                "status": fit_status,
            }
        )

pca_period_fit_summary_df = pd.DataFrame(pca_period_fit_records)

display(
    pca_period_fit_summary_df.assign(
        pc1_variance_pct=lambda df: df["pc1_variance_pct"].round(3),
        pc2_variance_pct=lambda df: df["pc2_variance_pct"].round(3),
        pc10_cumulative_variance_pct=lambda df: df["pc10_cumulative_variance_pct"].round(3),
    )
)

print("Pre/post PCA models ready.")

,feature_set,policy_period,rows,features,pc1_variance_pct,pc2_variance_pct,pc10_cumulative_variance_pct,status
0,bus,pre_cp,912228,40,18.025,11.643,71.203,loaded_from_cache
1,bus,post_cp,560719,40,17.376,10.971,70.478,loaded_from_cache
2,subway,pre_cp,569545,21,17.487,11.343,79.889,loaded_from_cache
3,subway,post_cp,341910,21,18.866,14.308,81.156,loaded_from_cache
4,taxi,pre_cp,955500,38,18.986,9.626,67.548,loaded_from_cache
5,taxi,post_cp,586300,38,21.142,8.325,68.360,loaded_from_cache
6,fhvhv,pre_cp,955500,37,21.967,10.593,72.998,loaded_from_cache
7,fhvhv,post_cp,586300,37,21.223,10.362,72.762,loaded_from_cache
8,multimodal,pre_cp,955500,142,12.920,6.756,43.654,loaded_from_cache
9,multimodal,post_cp,586300,142,13.177,6.608,43.552,loaded_from_cache


Pre/post PCA models ready.


Findings\. The pre/post PCA fits show broadly similar variance structure across all five feature sets: first\-10 cumulative variance changes by less than 1\.3 percentage points in every modality, suggesting that the overall dimensional complexity of each feature space remains stable across policy periods\.

Compare the number of components needed to reach common cumulative\-variance thresholds\. Stable component counts indicate that each feature space requires a similar number of PCA dimensions in the pre and post periods\.

In [8]:
# ---------------------------------------------------------------------
# Compare variance-threshold component counts across policy periods
# ---------------------------------------------------------------------

threshold_metric_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    for threshold in PCA_VARIANCE_THRESHOLDS:
        threshold_pct = int(threshold * 100)

        pre_components = components_for_threshold(
            PERIOD_PCA_CUMULATIVE_VARIANCE_BY_SET[feature_set_name]["pre_cp"],
            threshold,
        )
        post_components = components_for_threshold(
            PERIOD_PCA_CUMULATIVE_VARIANCE_BY_SET[feature_set_name]["post_cp"],
            threshold,
        )

        threshold_metric_records.append(
            {
                "feature_set": feature_set_name,
                "threshold": f"{threshold_pct}%",
                "pre_components": pre_components,
                "post_components": post_components,
                "component_delta": post_components - pre_components,
            }
        )

pca_period_threshold_comparison_df = pd.DataFrame(threshold_metric_records)

display(
    pca_period_threshold_comparison_df.sort_values(["feature_set", "threshold"])
)

print("Variance-threshold component counts compared across policy periods.")

,feature_set,threshold,pre_components,post_components,component_delta
0,bus,80%,14,14,0
1,bus,90%,19,19,0
2,bus,95%,23,23,0
9,fhvhv,80%,13,13,0
10,fhvhv,90%,18,19,1
11,fhvhv,95%,22,22,0
12,multimodal,80%,45,45,0
13,multimodal,90%,67,66,-1
14,multimodal,95%,84,83,-1
3,subway,80%,11,10,-1


Variance-threshold component counts compared across policy periods.


Findings\. The variance\-threshold counts are highly stable across policy periods: Bus and Taxi are identical at all thresholds, Subway and Multimodal shift by only one component at selected thresholds, and FHVHV shifts by one component only at 90%, reinforcing that pre/post feature\-space complexity is broadly unchanged\.

## 3\.2\.2\.3 Compare Principal Component Stability

This section checks whether the directions of that variance are also stable\. For each feature set, we compare the pre\_cp and post\_cp loading vectors for PC1, PC2, and PC3 using sign\-aligned cosine similarity and sign\-aligned loading correlation\. Because PCA signs are arbitrary, a component can point in the opposite direction and still represent the same feature relationship\.

Compare each pre\_cp principal component with its matching post\_cp component\. For example, Bus PC1 is compared against Bus post\_cp PC1, Bus PC2 against Bus post\_cp PC2, and so on through PC3\. The comparison uses the full feature\-loading vector for each component, not the PCA scores\. Values near 1 mean the same features define that component in both periods\. The sign\_alignment column records whether the post\_cp component had to be flipped because PCA component signs are arbitrary\.

In [9]:
# ---------------------------------------------------------------------
# Compare pre/post principal-component loading stability
# ---------------------------------------------------------------------

component_stability_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    pre_model = PERIOD_PCA_MODELS_BY_SET[feature_set_name]["pre_cp"]
    post_model = PERIOD_PCA_MODELS_BY_SET[feature_set_name]["post_cp"]

    for component_index in range(PCA_COMPONENTS_TO_COMPARE):
        pre_loading_vector = pre_model.components_[component_index]
        post_loading_vector = post_model.components_[component_index]

        (
            aligned_cosine_similarity,
            aligned_loading_correlation,
            sign_alignment,
            _,
        ) = sign_aligned_similarity(pre_loading_vector, post_loading_vector)

        component_stability_records.append(
            {
                "feature_set": feature_set_name,
                "component": f"PC{component_index + 1}",
                "aligned_cosine_similarity": aligned_cosine_similarity,
                "aligned_loading_correlation": aligned_loading_correlation,
                "sign_alignment": sign_alignment,
            }
        )

pca_component_stability_df = pd.DataFrame(component_stability_records)

display(
    pca_component_stability_df.assign(
        aligned_cosine_similarity=lambda df: df["aligned_cosine_similarity"].round(3),
        aligned_loading_correlation=lambda df: df["aligned_loading_correlation"].round(3),
    ).sort_values(["feature_set", "component"])
)

print("Pre/post principal-component stability compared.")

,feature_set,component,aligned_cosine_similarity,aligned_loading_correlation,sign_alignment
0,bus,PC1,0.998,0.998,1
1,bus,PC2,0.995,0.994,1
2,bus,PC3,0.984,0.984,1
9,fhvhv,PC1,0.999,0.999,1
10,fhvhv,PC2,0.999,0.998,1
11,fhvhv,PC3,0.982,0.985,-1
12,multimodal,PC1,0.993,0.993,1
13,multimodal,PC2,0.991,0.992,-1
14,multimodal,PC3,0.987,0.982,1
3,subway,PC1,0.927,0.919,-1


Pre/post principal-component stability compared.


Let's visualize the component\-stability scores as a compact heatmap\. Darker values indicate stronger alignment between the matching pre\_cp and post\_cp loading vectors\.

In [12]:
# ---------------------------------------------------------------------
# Visualize component-stability scores
# ---------------------------------------------------------------------

from project_branding import BRAND_COLORS

component_stability_heatmap_df = pca_component_stability_df.copy()

component_stability_heatmap_df["aligned_cosine_similarity"] = (
    component_stability_heatmap_df["aligned_cosine_similarity"].round(3)
)

component_stability_matrix_df = (
    component_stability_heatmap_df
    .pivot_table(
        index="feature_set",
        columns="component",
        values="aligned_cosine_similarity",
        aggfunc="first",
    )
    .reindex(index=MODEL_FEATURE_SET_NAMES)
)

component_stability_fig = px.imshow(
    component_stability_matrix_df,
    text_auto=".3f",
    aspect="auto",
    color_continuous_scale=[
        BRAND_COLORS["ice"],
        BRAND_COLORS["seafoam"],
        BRAND_COLORS["dark_teal"],
    ],
    zmin=0,
    zmax=1,
    labels={
        "x": "Principal component",
        "y": "Feature set",
        "color": "Aligned cosine similarity",
    },
    title="Pre/Post PCA Loading Stability by Feature Set",
)

component_stability_fig.update_layout(
    template="plotly_white",
    paper_bgcolor="white",
    plot_bgcolor=BRAND_COLORS["ice"],
    font={"color": BRAND_COLORS["dark_teal"]},
    title={"font": {"color": BRAND_COLORS["dark_teal"]}},
    width=850,
    height=430,
    margin={"l": 90, "r": 40, "t": 80, "b": 70},
    coloraxis_colorbar={
        "title": {
            "text": "Similarity",
            "font": {"color": BRAND_COLORS["dark_teal"]},
        },
        "tickfont": {"color": BRAND_COLORS["dark_teal"]},
    },
)

component_stability_fig.update_xaxes(
    side="top",
    tickfont={"color": BRAND_COLORS["dark_teal"]},
    title_font={"color": BRAND_COLORS["dark_teal"]},
)

component_stability_fig.update_yaxes(
    tickfont={"color": BRAND_COLORS["dark_teal"]},
    title_font={"color": BRAND_COLORS["dark_teal"]},
)

component_stability_fig.show()

print("Component-stability heatmap rendered.")

Component-stability heatmap rendered.


Findings\. This comparison checks whether each pre\_cp component is defined by the same feature\-loading pattern as its matching post\_cp component\. Bus, FHVHV, and Multimodal are highly stable across PC1\-PC3, meaning the same dominant feature relationships define those components in both periods\. Subway is moderately stable, especially by PC3\. Taxi is the main exception: PC1 remains stable, but PC2 and PC3 change substantially, suggesting that the secondary Taxi PCA structure shifted after congestion pricing\.

Summarize component stability by feature set\. This makes it easier to identify whether any modality has a broad stability issue across the leading components\.

In [14]:
# ---------------------------------------------------------------------
# Summarize component stability by feature set
# ---------------------------------------------------------------------

component_stability_summary_df = (
    pca_component_stability_df
    .groupby("feature_set", as_index=False)
    .agg(
        min_cosine_similarity=("aligned_cosine_similarity", "min"),
        median_cosine_similarity=("aligned_cosine_similarity", "median"),
        min_loading_correlation=("aligned_loading_correlation", "min"),
        median_loading_correlation=("aligned_loading_correlation", "median"),
    )
)

component_stability_summary_df["stability_status"] = np.select(
    [
        component_stability_summary_df["min_cosine_similarity"].ge(0.90),
        component_stability_summary_df["min_cosine_similarity"].ge(0.75),
    ],
    [
        "highly_stable",
        "moderately_stable",
    ],
    default="review",
)

display(
    component_stability_summary_df.assign(
        min_cosine_similarity=lambda df: df["min_cosine_similarity"].round(3),
        median_cosine_similarity=lambda df: df["median_cosine_similarity"].round(3),
        min_loading_correlation=lambda df: df["min_loading_correlation"].round(3),
        median_loading_correlation=lambda df: df["median_loading_correlation"].round(3),
    ).sort_values("min_cosine_similarity")
)

print("Component stability summarized by feature set.")

,feature_set,min_cosine_similarity,median_cosine_similarity,min_loading_correlation,median_loading_correlation,stability_status
4,taxi,0.318,0.686,0.226,0.558,review
3,subway,0.755,0.839,0.734,0.850,moderately_stable
1,fhvhv,0.982,0.999,0.985,0.998,highly_stable
0,bus,0.984,0.995,0.984,0.994,highly_stable
2,multimodal,0.987,0.991,0.982,0.992,highly_stable


Component stability summarized by feature set.


Findings\. The summary confirms that Bus, FHVHV, and Multimodal have highly stable leading PCA structures, Subway is moderately stable, and Taxi is the only feature set needing review because its PC2/PC3 loading patterns shift substantially across policy periods\.

### Section Summary

The component\-stability check identifies Taxi as the only feature set requiring closer inspection\. The next section uses leading\-feature overlap to determine whether the Taxi shift reflects a meaningful change in feature drivers or a reordering of secondary PCA components\.

## 3\.2\.2\.4 Compare Leading Feature Drivers

The stability scores identify where pre\_cp and post\_cp components remain aligned, but they do not show which features are responsible\. This section compares the strongest absolute\-loading features behind each leading component\. If the same features dominate a component in both periods, the PCA axis is not only numerically stable but also interpretable in the same way\. If overlap drops, the feature lists help show whether the change reflects a new mobility relationship or a reordering of secondary component structure\.

Create top\-loading feature lists for each feature set, policy period, and leading component\. Features are ranked by absolute loading because loading magnitude shows how strongly a feature contributes to the component, regardless of sign\.

In [15]:
# ---------------------------------------------------------------------
# Rank leading absolute-loading features by period and component
# ---------------------------------------------------------------------

period_top_loading_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    feature_names = PCA_INPUT_COLUMNS_BY_SET[feature_set_name]

    for period_value in POLICY_PERIOD_VALUES:
        pca_model = PERIOD_PCA_MODELS_BY_SET[feature_set_name][period_value]

        for component_index in range(PCA_COMPONENTS_TO_COMPARE):
            component_name = f"PC{component_index + 1}"
            component_loadings = pca_model.components_[component_index]

            component_loading_df = pd.DataFrame(
                {
                    "feature_name": feature_names,
                    "loading": component_loadings,
                    "abs_loading": np.abs(component_loadings),
                }
            ).sort_values("abs_loading", ascending=False)

            for rank, (_, loading_row) in enumerate(
                component_loading_df.head(PCA_TOP_FEATURE_COUNT).iterrows(),
                start=1,
            ):
                period_top_loading_records.append(
                    {
                        "feature_set": feature_set_name,
                        "policy_period": period_value,
                        "component": component_name,
                        "rank": rank,
                        "feature_name": loading_row["feature_name"],
                        "loading": loading_row["loading"],
                        "abs_loading": loading_row["abs_loading"],
                    }
                )

period_top_loading_df = pd.DataFrame(period_top_loading_records)

top_loading_count_summary_df = (
    period_top_loading_df
    .groupby(["feature_set", "policy_period", "component"], as_index=False)
    .agg(top_feature_count=("feature_name", "nunique"))
)

display(top_loading_count_summary_df)

print("Leading absolute-loading features ranked by period and component.")

,feature_set,policy_period,component,top_feature_count
0,bus,post_cp,PC1,10
1,bus,post_cp,PC2,10
2,bus,post_cp,PC3,10
3,bus,pre_cp,PC1,10
4,bus,pre_cp,PC2,10
5,bus,pre_cp,PC3,10
6,fhvhv,post_cp,PC1,10
7,fhvhv,post_cp,PC2,10
8,fhvhv,post_cp,PC3,10
9,fhvhv,pre_cp,PC1,10


Leading absolute-loading features ranked by period and component.


Findings\. Each feature set, period, and leading component has the expected 10 ranked loading features, so the overlap comparison is using a complete and balanced top\-feature list\.

Summarize how much the top\-loading feature sets overlap between pre\_cp and post\_cp\. Low overlap marks components that need closer inspection; high overlap means the same features remain dominant across periods\.

In [16]:
# ---------------------------------------------------------------------
# Compare pre/post top-loading feature overlap
# ---------------------------------------------------------------------

top_loading_overlap_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    for component_index in range(PCA_COMPONENTS_TO_COMPARE):
        component_name = f"PC{component_index + 1}"

        pre_features = set(
            period_top_loading_df.loc[
                period_top_loading_df["feature_set"].eq(feature_set_name)
                & period_top_loading_df["policy_period"].eq("pre_cp")
                & period_top_loading_df["component"].eq(component_name),
                "feature_name",
            ]
        )

        post_features = set(
            period_top_loading_df.loc[
                period_top_loading_df["feature_set"].eq(feature_set_name)
                & period_top_loading_df["policy_period"].eq("post_cp")
                & period_top_loading_df["component"].eq(component_name),
                "feature_name",
            ]
        )

        shared_features = sorted(pre_features & post_features)

        top_loading_overlap_records.append(
            {
                "feature_set": feature_set_name,
                "component": component_name,
                "pre_top_features": len(pre_features),
                "post_top_features": len(post_features),
                "shared_top_features": len(shared_features),
                "top_feature_overlap_pct": len(shared_features) / PCA_TOP_FEATURE_COUNT,
                "shared_feature_examples": ", ".join(shared_features[:3]),
            }
        )

top_loading_overlap_df = pd.DataFrame(top_loading_overlap_records)

top_loading_overlap_df["overlap_status"] = np.select(
    [
        top_loading_overlap_df["top_feature_overlap_pct"].ge(0.80),
        top_loading_overlap_df["top_feature_overlap_pct"].ge(0.50),
    ],
    [
        "high_overlap",
        "moderate_overlap",
    ],
    default="review",
)

top_loading_overlap_display_df = (
    top_loading_overlap_df[
        [
            "feature_set",
            "component",
            "shared_top_features",
            "top_feature_overlap_pct",
            "overlap_status",
        ]
    ]
    .assign(
        top_feature_overlap_pct=lambda df: df["top_feature_overlap_pct"].round(2)
    )
    .sort_values(["overlap_status", "feature_set", "component"])
    .reset_index(drop=True)
)

display(top_loading_overlap_display_df)

print("Pre/post top-loading feature overlap compared.")

,feature_set,component,shared_top_features,top_feature_overlap_pct,overlap_status
0,bus,PC1,10,1.0,high_overlap
1,bus,PC2,9,0.9,high_overlap
2,bus,PC3,10,1.0,high_overlap
3,fhvhv,PC1,10,1.0,high_overlap
4,fhvhv,PC2,10,1.0,high_overlap
5,fhvhv,PC3,9,0.9,high_overlap
6,multimodal,PC3,9,0.9,high_overlap
7,subway,PC1,9,0.9,high_overlap
8,taxi,PC1,10,1.0,high_overlap
9,multimodal,PC1,5,0.5,moderate_overlap


Pre/post top-loading feature overlap compared.


In [17]:
# ---------------------------------------------------------------------
# Inspect Taxi review-case feature drivers side by side
# ---------------------------------------------------------------------

REVIEW_FEATURE_SET = "taxi"
REVIEW_COMPONENTS = ["PC2", "PC3"]

taxi_review_driver_df = (
    period_top_loading_df
    .loc[
        period_top_loading_df["feature_set"].eq(REVIEW_FEATURE_SET)
        & period_top_loading_df["component"].isin(REVIEW_COMPONENTS)
    ]
    .copy()
)

taxi_review_driver_df["feature_label"] = (
    taxi_review_driver_df["feature_name"]
    .str.replace("taxi_", "", regex=False)
    .str.replace("_same_bucket", "", regex=False)
    .str.replace("_", " ", regex=False)
)

taxi_review_side_by_side_df = (
    taxi_review_driver_df
    .assign(
        ranked_feature=lambda df: (
            df["rank"].astype(str)
            + ". "
            + df["feature_label"]
            + " ("
            + df["loading"].round(3).astype(str)
            + ")"
        )
    )
    .pivot_table(
        index=["component", "rank"],
        columns="policy_period",
        values="ranked_feature",
        aggfunc="first",
    )
    .reset_index()
    .sort_values(["component", "rank"])
)

taxi_review_side_by_side_df = taxi_review_side_by_side_df[
    ["component", "rank", "pre_cp", "post_cp"]
]

display(taxi_review_side_by_side_df)

print("Taxi review-case feature drivers inspected side by side.")

policy_period,component,rank,pre_cp,post_cp
0,PC2,1,1. avg trip speed residual (0.484),1. avg trip speed (0.36)
1,PC2,2,2. avg trip speed residual zscore (0.459),2. borough mean avg trip speed (0.31)
2,PC2,3,3. avg trip speed (0.395),3. avg trip speed post cp mean (0.305)
3,PC2,4,4. avg trip speed fourier20 residual reconstru...,4. avg trip speed lag 1 (0.256)
4,PC2,5,5. avg trip distance (0.272),5. avg trip speed pre cp mean (0.243)
5,PC2,6,6. avg trip speed residual abs (0.254),6. trip count rolling std 7 (0.239)
6,PC2,7,7. avg trip speed deviation abs zscore (0.187),7. trip count rolling mean 7 (0.22)
7,PC2,8,8. trip count residual abs (0.109),8. trip count lag 1 (0.22)
8,PC2,9,9. trip count rolling std 7 (0.107),9. cp zone mean trip count (-0.204)
9,PC2,10,10. trip count rolling mean 7 (0.102),10. trip count (0.196)


Taxi review-case feature drivers inspected side by side.


Findings\. The Taxi review table shows that the instability is concentrated in the secondary components rather than Taxi PC1\. Taxi PC2 stays partly speed\-oriented, but its post\_cp version shifts away from residual/deviation speed features toward current speed, borough speed context, pre/post speed means, and demand\-history features\. Taxi PC3 changes more sharply: pre\_cp PC3 is mostly speed baseline, distance, and demand\-history structure, while post\_cp PC3 is dominated by trip\-count residuals, changes, Fourier reconstruction, and seasonal demand features\. This suggests that Taxi’s primary activity axis remains stable, but its secondary PCA structure changes from speed/spatial context toward demand\-deviation behavior after congestion pricing\.

Visualize which top\-loading features appear in the pre\_cp and post\_cp versions of PC1\-PC3\. Each tile marks whether a feature is among the top\-loading drivers for that period/component\. Shared rows indicate stable feature drivers; one\-sided rows show where the feature drivers diverge\.

In [18]:
# ---------------------------------------------------------------------
# Visualize top-feature intersection and divergence by component
# ---------------------------------------------------------------------

TOP_FEATURE_MEMBERSHIP_COMPONENTS = ["PC1", "PC2", "PC3"]

top_feature_membership_df = (
    period_top_loading_df
    .loc[period_top_loading_df["component"].isin(TOP_FEATURE_MEMBERSHIP_COMPONENTS)]
    .copy()
)

top_feature_membership_df["period_component"] = (
    top_feature_membership_df["policy_period"].map(
        {
            "pre_cp": "Pre",
            "post_cp": "Post",
        }
    )
    + " "
    + top_feature_membership_df["component"]
)

top_feature_membership_df["feature_label"] = (
    top_feature_membership_df["feature_name"]
    .str.replace("_same_bucket", "", regex=False)
    .str.replace("_missing_indicator", " missing", regex=False)
    .str.replace("_", " ", regex=False)
)

period_component_order = [
    "Pre PC1",
    "Post PC1",
    "Pre PC2",
    "Post PC2",
    "Pre PC3",
    "Post PC3",
]

membership_fig = go.Figure()
membership_buttons = []

for feature_set_index, feature_set_name in enumerate(MODEL_FEATURE_SET_NAMES):
    feature_set_membership_df = top_feature_membership_df.loc[
        top_feature_membership_df["feature_set"].eq(feature_set_name)
    ].copy()

    feature_order_df = (
        feature_set_membership_df
        .groupby("feature_label", as_index=False)
        .agg(
            membership_count=("period_component", "nunique"),
            best_rank=("rank", "min"),
        )
        .sort_values(
            ["membership_count", "best_rank", "feature_label"],
            ascending=[False, True, True],
        )
    )

    feature_order = feature_order_df["feature_label"].tolist()

    feature_membership_matrix_df = (
        feature_set_membership_df
        .assign(present=1)
        .pivot_table(
            index="feature_label",
            columns="period_component",
            values="present",
            aggfunc="max",
            fill_value=0,
        )
        .reindex(
            index=feature_order,
            columns=period_component_order,
            fill_value=0,
        )
    )

    hover_text = []

    for feature_label in feature_membership_matrix_df.index:
        hover_row = []
        for period_component in feature_membership_matrix_df.columns:
            matching_rows = feature_set_membership_df.loc[
                feature_set_membership_df["feature_label"].eq(feature_label)
                & feature_set_membership_df["period_component"].eq(period_component)
            ]

            if matching_rows.empty:
                hover_row.append(
                    f"Feature: {feature_label}<br>{period_component}: not in top {PCA_TOP_FEATURE_COUNT}"
                )
            else:
                row = matching_rows.iloc[0]
                hover_row.append(
                    f"Feature: {feature_label}<br>"
                    f"{period_component}<br>"
                    f"Rank: {int(row['rank'])}<br>"
                    f"Loading: {row['loading']:.3f}"
                )

        hover_text.append(hover_row)

    membership_fig.add_trace(
        go.Heatmap(
            z=feature_membership_matrix_df.values,
            x=feature_membership_matrix_df.columns.tolist(),
            y=feature_membership_matrix_df.index.tolist(),
            customdata=hover_text,
            colorscale=[
                [0, BRAND_COLORS["ice"]],
                [1, BRAND_COLORS["dark_teal"]],
            ],
            zmin=0,
            zmax=1,
            showscale=False,
            visible=(feature_set_index == 0),
            hovertemplate="%{customdata}<extra></extra>",
            xgap=2,
            ygap=2,
        )
    )

    visibility = [False] * len(MODEL_FEATURE_SET_NAMES)
    visibility[feature_set_index] = True

    membership_buttons.append(
        {
            "label": feature_set_name,
            "method": "update",
            "args": [
                {"visible": visibility},
                {"title": f"Top PCA Feature Driver Membership: {feature_set_name}"},
            ],
        }
    )

membership_fig.update_layout(
    title=f"Top PCA Feature Driver Membership: {MODEL_FEATURE_SET_NAMES[0]}",
    width=900,
    height=650,
    template="plotly_white",
    paper_bgcolor="white",
    plot_bgcolor=BRAND_COLORS["ice"],
    font={"color": BRAND_COLORS["dark_teal"]},
    margin={"l": 360, "r": 40, "t": 90, "b": 90},
    xaxis_title="Period and component",
    yaxis_title=None,
    updatemenus=[
        {
            "buttons": membership_buttons,
            "direction": "down",
            "showactive": True,
            "x": 1.0,
            "xanchor": "right",
            "y": 1.12,
            "yanchor": "top",
            "bgcolor": "white",
            "bordercolor": BRAND_COLORS["seafoam"],
            "font": {"color": BRAND_COLORS["dark_teal"]},
        }
    ],
)

membership_fig.update_xaxes(
    side="top",
    tickfont={"color": BRAND_COLORS["dark_teal"]},
    title_font={"color": BRAND_COLORS["dark_teal"]},
    showgrid=False,
)

membership_fig.update_yaxes(
    autorange="reversed",
    tickfont={"size": 10, "color": BRAND_COLORS["dark_teal"]},
    automargin=True,
    ticklabelstandoff=18,
    showgrid=False,
)

membership_fig.update_layout(
    shapes=[
        {
            "type": "line",
            "xref": "x",
            "yref": "paper",
            "x0": 1.5,
            "x1": 1.5,
            "y0": 0,
            "y1": 1,
            "line": {
                "color": BRAND_COLORS["terracotta"],
                "width": 1.5,
            },
        },
        {
            "type": "line",
            "xref": "x",
            "yref": "paper",
            "x0": 3.5,
            "x1": 3.5,
            "y0": 0,
            "y1": 1,
            "line": {
                "color": BRAND_COLORS["terracotta"],
                "width": 1.5,
            },
        },
    ]
)

membership_fig.show()

print("Top-feature membership heatmap rendered.")

Top-feature membership heatmap rendered.


Findings\. The feature\-driver membership view shows that the main pre/post stability story is not uniform across modalities: Bus and FHVHV preserve very consistent driver sets, Subway and Multimodal partially reshuffle secondary components, and Taxi shows the clearest divergence after PC1\.

- Bus is highly stable across all three components\. PC1 and PC2 keep the same demand, speed\-state, distance, and CBD\-context drivers across periods, while PC3 also preserves the same change/residual feature family\. This supports treating Bus PCA structure as stable pre/post\.

- Subway PC1 is stable, with ridership, spatial distance, connected\-zone, and neighbor\-count features appearing in both periods\. PC2 and PC3 show more reshuffling across ridership residuals, transfers, seasonal/deviation features, and CBD/gateway context, which matches the moderate stability signal from the similarity metrics\.

- Taxi PC1 is stable, but PC2 and PC3 change substantially\. Pre/post PC1 retains trip\-count, rolling demand, distance, and CBD\-context drivers, while PC2 and PC3 shift from speed residual/baseline structure toward a stronger mix of speed\-state and trip\-count residual/change features\. Taxi is the clearest case where secondary PCA structure changes across periods\.

- FHVHV is the cleanest stability case\. PC1 and PC2 preserve the same speed, demand, trip\-count, and trip\-distance drivers across periods, and PC3 remains concentrated in residual/change features\. The top\-driver pattern supports the high cosine and high overlap results\.

- Multimodal shows stable themes but less one\-to\-one feature continuity\. Demand and cross\-modal features remain important, but PC1 and PC2 redistribute some Taxi, FHVHV, Bus, and interaction drivers across periods\. This suggests the broad Multimodal structure is stable, while individual component membership is more flexible than in the single\-modality Bus/FHVHV spaces\.

> ▣ Decision\. For downstream PCA\-based anomaly detection, Bus, FHVHV, Subway, and Multimodal can use full\-period PCA representations because their leading component structures are stable across pre\_cp and post\_cp observations\. Taxi should use a post\_cp\-only PCA representation when the analysis focuses on post\-congestion\-pricing anomalies, because Taxi PC2 and PC3 change meaning across periods\. Full\-period Taxi PCA remains useful for historical comparison, but post\-CP anomaly scoring should use the post\-CP Taxi feature\-space structure as its reference\.

### Section Summary

The pre/post stability analysis suggests that congestion pricing did not fundamentally reorganize the dominant mobility feature space\. Bus, FHVHV, and Multimodal retain highly similar leading component directions, and Subway remains moderately stable with some reshuffling in secondary components\. This supports using a shared PCA\-based representation across both policy periods rather than fitting separate pre/post representations by default\.

Taxi is the main exception\. Taxi PC1 remains stable, so the primary Taxi activity structure can be compared across periods\. Taxi PC2 and PC3 shift substantially in both loading\-vector similarity and top\-feature membership, meaning those secondary components do not carry the same interpretation before and after congestion pricing\. In later clustering or anomaly\-detection notebooks, Taxi patterns driven by PC2 or PC3 should be interpreted as possible changes in the secondary Taxi mobility structure, not simply as movement along a fixed pre/post axis\.

## 3\.2\.2\.5 Export Taxi Period\-Specific PCA Reference Assets

Taxi is the only feature set where the pre/post PCA stability analysis surfaced a meaningful shift in secondary component structure\. Rather than forcing later anomaly notebooks to recompute those period\-specific Taxi PCA spaces, this section exports them now as reusable reference assets\.

These exports do not replace the canonical full\-period Taxi PCA representation from 3\.2\.1\. They sit alongside it as targeted downstream assets so later notebooks can directly compare shared Taxi PCA against split pre/post Taxi PCA when needed\.

### Define Taxi period\-specific PCA export targets

Before writing anything to disk, let’s make the Taxi export contract explicit\. We’ll export one pre\-CP PCA score table and one post\-CP PCA score table, both using the 80% cumulative explained variance rule so they line up with the downstream PCA representation logic established in 3\.2\.4\.

In [19]:
# ---------------------------------------------------------------------
# Define Taxi period-specific PCA export targets
# ---------------------------------------------------------------------

TAXI_PERIOD_PCA_EXPORT_THRESHOLD = 0.80

TAXI_PERIOD_PCA_OUTPUT_PATHS = {
    period_value: OUTPUT_DIR / f"taxi_{period_value}_pca_80pct_modeling_scores.parquet"
    for period_value in POLICY_PERIOD_VALUES
}

TAXI_PERIOD_PCA_SUMMARY_PATH = (
    OUTPUT_DIR / "taxi_period_specific_pca_export_summary.csv"
)

taxi_period_export_target_records = []

for period_value in POLICY_PERIOD_VALUES:
    expected_components = components_for_threshold(
        PERIOD_PCA_CUMULATIVE_VARIANCE_BY_SET["taxi"][period_value],
        TAXI_PERIOD_PCA_EXPORT_THRESHOLD,
    )

    taxi_period_export_target_records.append(
        {
            "policy_period": period_value,
            "representation_name": "taxi_pca_80pct",
            "expected_components": expected_components,
            "output_path": str(TAXI_PERIOD_PCA_OUTPUT_PATHS[period_value]),
            "path_exists": TAXI_PERIOD_PCA_OUTPUT_PATHS[period_value].exists(),
        }
    )

taxi_period_export_targets_df = pd.DataFrame(taxi_period_export_target_records)

display(taxi_period_export_targets_df)

print("Taxi period-specific PCA export targets defined.")

,policy_period,representation_name,expected_components,output_path,path_exists
0,pre_cp,taxi_pca_80pct,15,pipeline_data/3.2.2.final_tables/taxi_pre_cp_p...,True
1,post_cp,taxi_pca_80pct,15,pipeline_data/3.2.2.final_tables/taxi_post_cp_...,True


Taxi period-specific PCA export targets defined.


### Materialize Taxi period\-specific PCA score assets

Now let’s write the actual Taxi pre/post PCA score tables\. Each export preserves modeling\_row\_id and writes only the PCA score columns needed to reach the 80% variance threshold for that period, so the outputs behave like downstream\-ready representation tables rather than mixed analysis tables\.

In [20]:
# ---------------------------------------------------------------------
# Materialize Taxi period-specific PCA score assets
# ---------------------------------------------------------------------

should_export_taxi_period_pca = (
    REBUILD_TAXI_PERIOD_PCA_EXPORTS
    or any(not output_path.exists() for output_path in TAXI_PERIOD_PCA_OUTPUT_PATHS.values())
)

taxi_period_pca_export_records = []

if should_export_taxi_period_pca:
    taxi_matrix_path_sql = sql_path(SCALED_MATRIX_PATHS_BY_SET["taxi"])
    taxi_metadata_path_sql = sql_path(ROW_METADATA_PATHS_BY_SET["taxi"])
    taxi_input_columns = PCA_INPUT_COLUMNS_BY_SET["taxi"]

    selected_columns_sql = ", ".join(
        duckdb_identifier(column_name) for column_name in taxi_input_columns
    )

    for period_value in POLICY_PERIOD_VALUES:
        pca_model = PERIOD_PCA_MODELS_BY_SET["taxi"][period_value]
        component_count = components_for_threshold(
            PERIOD_PCA_CUMULATIVE_VARIANCE_BY_SET["taxi"][period_value],
            TAXI_PERIOD_PCA_EXPORT_THRESHOLD,
        )

        with duckdb.connect() as con:
            taxi_period_input_df = con.execute(
                f"""
                SELECT
                    x.modeling_row_id,
                    {selected_columns_sql}
                FROM read_parquet('{taxi_matrix_path_sql}') AS x
                INNER JOIN read_parquet('{taxi_metadata_path_sql}') AS m
                    USING (modeling_row_id)
                WHERE m.pre_post_cp = '{period_value}'
                ORDER BY x.modeling_row_id
                """
            ).fetchdf()

        modeling_row_id_series = taxi_period_input_df["modeling_row_id"].copy()
        X = taxi_period_input_df[taxi_input_columns].astype("float32", copy=False)

        # Transform with the period-specific Taxi PCA model, then keep only the
        # leading components required to reach the 80% variance threshold.
        taxi_period_scores = pca_model.transform(X)[:, :component_count]

        taxi_period_score_df = pd.DataFrame(
            taxi_period_scores,
            columns=[f"PC{i + 1}" for i in range(component_count)],
        )
        taxi_period_score_df.insert(0, "modeling_row_id", modeling_row_id_series.to_numpy())

        taxi_period_score_df.to_parquet(
            TAXI_PERIOD_PCA_OUTPUT_PATHS[period_value],
            index=False,
        )

        taxi_period_pca_export_records.append(
            {
                "policy_period": period_value,
                "rows_written": len(taxi_period_score_df),
                "components_written": component_count,
                "output_path": str(TAXI_PERIOD_PCA_OUTPUT_PATHS[period_value]),
                "output_action": "written",
                "status": "pass",
            }
        )
else:
    for period_value in POLICY_PERIOD_VALUES:
        exported_df = pd.read_parquet(TAXI_PERIOD_PCA_OUTPUT_PATHS[period_value])

        taxi_period_pca_export_records.append(
            {
                "policy_period": period_value,
                "rows_written": len(exported_df),
                "components_written": len(
                    [column_name for column_name in exported_df.columns if column_name != "modeling_row_id"]
                ),
                "output_path": str(TAXI_PERIOD_PCA_OUTPUT_PATHS[period_value]),
                "output_action": "loaded_existing",
                "status": "pass",
            }
        )

taxi_period_pca_export_summary_df = pd.DataFrame(taxi_period_pca_export_records)
taxi_period_pca_export_summary_df.to_csv(
    TAXI_PERIOD_PCA_SUMMARY_PATH,
    index=False,
)

display(taxi_period_pca_export_summary_df)

assert taxi_period_pca_export_summary_df["status"].eq("pass").all(), (
    "Taxi period-specific PCA exports did not materialize successfully."
)

print("Taxi period-specific PCA score assets materialized.")

,policy_period,rows_written,components_written,output_path,output_action,status
0,pre_cp,955500,15,pipeline_data/3.2.2.final_tables/taxi_pre_cp_p...,loaded_existing,pass
1,post_cp,586300,15,pipeline_data/3.2.2.final_tables/taxi_post_cp_...,loaded_existing,pass


Taxi period-specific PCA score assets materialized.


### Validate exported Taxi period\-specific PCA assets

With the files written, the last step is a compact QA pass\. We want to confirm that each Taxi period\-specific PCA export has the expected row count, the expected number of 80%\-threshold components, unique row ids, and no missing score values\.

In [21]:
# ---------------------------------------------------------------------
# Validate exported Taxi period-specific PCA assets
# ---------------------------------------------------------------------

taxi_period_pca_validation_records = []

for period_value in POLICY_PERIOD_VALUES:
    exported_df = pd.read_parquet(TAXI_PERIOD_PCA_OUTPUT_PATHS[period_value])

    expected_rows = int(
        period_split_summary_df.loc[
            period_split_summary_df["feature_set"].eq("taxi")
            & period_split_summary_df["pre_post_cp"].eq(period_value),
            "rows",
        ].iloc[0]
    )

    expected_components = components_for_threshold(
        PERIOD_PCA_CUMULATIVE_VARIANCE_BY_SET["taxi"][period_value],
        TAXI_PERIOD_PCA_EXPORT_THRESHOLD,
    )

    score_columns = [
        column_name
        for column_name in exported_df.columns
        if column_name != "modeling_row_id"
    ]

    duplicate_modeling_rows = int(exported_df["modeling_row_id"].duplicated().sum())
    null_score_cells = int(exported_df[score_columns].isna().sum().sum())

    taxi_period_pca_validation_records.append(
        {
            "policy_period": period_value,
            "export_rows": len(exported_df),
            "expected_rows": expected_rows,
            "pca_component_columns": len(score_columns),
            "expected_components": expected_components,
            "duplicate_modeling_rows": duplicate_modeling_rows,
            "null_score_cells": null_score_cells,
            "status": "pass"
            if (
                len(exported_df) == expected_rows
                and len(score_columns) == expected_components
                and duplicate_modeling_rows == 0
                and null_score_cells == 0
            )
            else "review",
        }
    )

taxi_period_pca_validation_df = pd.DataFrame(taxi_period_pca_validation_records)

display(taxi_period_pca_validation_df)

assert taxi_period_pca_validation_df["status"].eq("pass").all(), (
    "One or more Taxi period-specific PCA exports failed validation."
)

print("Taxi period-specific PCA exports validated.")

,policy_period,export_rows,expected_rows,pca_component_columns,expected_components,duplicate_modeling_rows,null_score_cells,status
0,pre_cp,955500,955500,15,15,0,0,pass
1,post_cp,586300,586300,15,15,0,0,pass


Taxi period-specific PCA exports validated.


Findings\. The Taxi period\-specific PCA exports validated cleanly: both pre\_cp and post\_cp score tables have the expected row counts, the expected 15\-component 80%\-variance representation, no duplicate modeling\_row\_id values, and no missing score cells\.

### Section Summary

These Taxi period\-specific PCA assets are now available as targeted downstream references\. Later anomaly notebooks can compare shared Taxi PCA against split pre/post Taxi PCA directly, without needing to rebuild period\-specific PCA spaces from scratch\.

## Close

Notebook 3\.2\.2 shows that the broad PCA feature\-space structure is stable across congestion\-pricing periods for most transportation systems\. Variance profiles are nearly unchanged, and Bus, FHVHV, and Multimodal retain highly similar leading component directions\. Subway shows moderate reshuffling in secondary components but remains stable enough for full\-period PCA use\.

Taxi is the main exception\. Taxi PC1 remains stable, but PC2 and PC3 change substantially in both loading similarity and feature\-driver membership\. For downstream PCA\-based anomaly detection, full\-period PCA remains appropriate for Bus, Subway, FHVHV, and Multimodal\. Taxi post\-CP anomaly workflows should instead consider a post\-CP\-only Taxi PCA reference so that secondary Taxi anomalies are scored against the post\-policy Taxi mobility structure rather than a mixed pre/post feature\-space definition\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>